# Overview

In [ ]:
chunk_start

In [ ]:
import datetime
from oceanval import sim_compare
from oceanval import sim_plots as sp
from oceanval import sim_report as sr
from oceanval import analysis_plots as ap

# what compare_simulations() saved, beside the report
results_dir = "../../results"
info = sim_compare.load(results_dir)
sims = [sim["name"] for sim in info["simulations"]]
by_name = {sim["name"]: sim for sim in info["simulations"]}
years_text = sr.years_text

In [ ]:
created = datetime.datetime.fromisoformat(info["created"])
count = len(info["variables"])
md(
    f"This report compares {len(sims)} simulations: {sr.colour_names(info)}, with no observations. "
    f"It covers {count} variable{'s' if count != 1 else ''}, and was made with **oceanval** "
    f"{info['oceanval']} on {created.day} {created.strftime('%B %Y')}. "
    "Each variable has a page of its own, listed on the left, in which the simulations are shown side by side, "
    "each in the colour it has in the tables below."
)

In [ ]:
rows = []
for sim in info["simulations"]:
    rows.append(
        {
            "Simulation": sr.swatch(info, sim["name"]),
            "Directory": sim["sim_dir"],
            "Directories down": sim["n_dirs_down"],
            "Thickness": sim["thickness"] or "None",
        }
    )
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
md(f"**Table {i_table}:** The simulations, the directories their output is in, and how many directories below those the files are.")
i_table += 1

In [ ]:
rows = []
for variable in info["variables"]:
    row = {"Variable": variable["name"], "Unit": ap.unit_html(variable["units"])}
    for sim in sims:
        own = variable["sims"][sim]
        row[sr.swatch(info, sim)] = f"<code>{own['model_variable'].replace('+', ' + ')}</code>"
    rows.append(row)
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
md(f"**Table {i_table}:** The variables compared, and the model variable holding each in each simulation.")
i_table += 1

In [ ]:
rows = []
for variable in info["variables"]:
    for sim in sims:
        own = variable["sims"][sim]
        rows.append(
            {
                "Variable": variable["name"],
                "Simulation": sr.swatch(info, sim),
                "Files": f"{own['pattern']} ({own['files']})",
                "Output": own["time_res_text"].capitalize(),
                "Dates": f"{own['first']} to {own['last']}",
            }
        )
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
md(f"**Table {i_table}:** The output each simulation's variables are read from: the files, named with their dates left out, and how many there are.")
i_table += 1

In [ ]:
rows = []
for variable in info["variables"]:
    own = variable["sims"][sims[0]]
    clim = own.get("climatology")
    phen = own.get("phenology")
    long_term = own.get("long_term")
    seasonal = own.get("seasonal")
    row = {
        "Variable": variable["name"],
        "Climatology": (
            f"{years_text(clim)}: " + ", ".join(layer["title"].lower() for layer in clim["layers"].values())
            if clim else "None"
        ),
        "Phenology": f"{years_text(phen)}: surface" if phen else "None",
    }
    if any("long_term" in x["kinds"] for x in info["variables"]):
        row["Long-term series"] = (
            f"{years_text(long_term)}: " + ", ".join(layer["title"].lower() for layer in long_term["layers"].values())
            if long_term else "None"
        )
    if any("seasonal" in x["kinds"] for x in info["variables"]):
        row["Seasonal cycles"] = (
            f"{years_text(seasonal)}: " + ", ".join(layer["title"].lower() for layer in seasonal["layers"].values())
            if seasonal else "None"
        )
    if any("map_changes" in x["kinds"] for x in info["variables"]):
        changes = own.get("map_changes")
        row["Map changes"] = (
            ", ".join(f"{period['start']}-{period['end']}" for period in changes["periods"])
            + ": " + ", ".join(layer["title"].lower() for layer in changes["layers"].values())
            if changes else "None"
        )
    if any("transects" in x["kinds"] for x in info["variables"]):
        transects = own.get("transects")
        row["Transects"] = f"{years_text(transects)}: {len(transects['lines'])}" if transects else "None"
    rows.append(row)
pd.DataFrame(rows).style.hide(axis="index")

In [ ]:
md(f"**Table {i_table}:** What was worked out for each variable in every simulation, and over which years.")
i_table += 1

In [ ]:
md("## Model grids")
# with a domain mask and no unify, the mask's grid is the one grid, as its sentence says
if info["unify"] is not None or not info.get("domain_mask_text"):
    md(info["unify_text"])
if info.get("domain_mask_text"):
    md(info["domain_mask_text"])
if info["unify"] is None:
    if info["same_grid"] is True:
        md("The simulations are on the same grid.")
    elif info["same_grid"] is False:
        md(
            "**The simulations are not all on the same grid**, and they were not put on one: the maps are of each simulation's own grid, "
            "and the means over the domain, the regions and the transects are over the cells each simulation has, "
            "so they are not over exactly the same area."
        )
    else:
        md("Whether the simulations are on the same grid could not be told.")
else:
    rows = []
    for variable in info["variables"]:
        for layer, (shared, cells) in (info["shared_cells"] or {}).get(variable["key"], {}).items():
            rows.append(
                {
                    "Variable": variable["name"],
                    "Layer": layer if layer == "levels" else ap.LAYER_TITLES[layer] if hasattr(ap, "LAYER_TITLES") else layer,
                    "Cells with values in every simulation": f"{shared} of {cells}",
                }
            )
    if rows:
        display(pd.DataFrame(rows).style.hide(axis="index"))
        md(f"**Table {i_table}:** The number of cells of the common grid with a value in every simulation, which are the cells all the simulations are compared over. A cell with no value in one simulation is missing in all of them.")
        i_table += 1

In [ ]:
notes = []
for variable in info["variables"]:
    for sim in sims:
        own = variable["sims"][sim]
        for kind in ("climatology", "phenology", "long_term", "seasonal", "transects"):
            analysis = own.get(kind)
            if analysis and analysis["dates"]["issues"]:
                notes.append(f"- **{variable['name']}**, {sim}, {kind.replace('_', '-')} ({years_text(analysis)}): " + " ".join(analysis["dates"]["issues"]))
        changes = own.get("map_changes")
        for period in changes["periods"] if changes else []:
            if period["dates"]["issues"]:
                notes.append(f"- **{variable['name']}**, {sim}, map changes, {period['title'].lower()} period ({years_text(period)}): " + " ".join(period["dates"]["issues"]))
md("## Gaps in the output")
if notes:
    md("The output is missing times in the years some of the variables were analysed for. Averages are taken over the times there are.\n\n" + "\n".join(notes))
else:
    md("No times are missing from the output of any simulation in the years each variable was analysed for.")

In [ ]:
md("## Methods")
paragraphs = sr.methods(info)
at = 1
if info["unify"] is not None or not info.get("domain_mask_text"):
    paragraphs.insert(at, "**Model grids.** " + info["unify_text"])
    at += 1
if info.get("domain_mask_text"):
    paragraphs.insert(at, "**Domain mask.** " + info["domain_mask_text"])
thicknesses = {sim["name"]: sim["thickness"] for sim in info["simulations"] if sim["thickness"]}
if thicknesses:
    paragraphs.append(
        "The cell thickness of the water column is "
        + ("on fixed depth levels (z-levels), whose thicknesses are taken from the levels themselves" if set(thicknesses.values()) == {"z_level"} else
           "; ".join(f"`{x}` in {sim}" if x != "z_level" else f"the levels themselves in {sim}" for sim, x in thicknesses.items()))
        + ", at its first time if it changes over time."
    )
missing = {sim["name"]: sim["as_missing"] for sim in info["simulations"] if sim["as_missing"] is not None}
if missing:
    def what(value):
        if isinstance(value, list):
            return f"{value[0]:g}" if value[0] == value[1] else f"{value[0]:g} to {value[1]:g}"
        return f"{value:g}"
    paragraphs.append(
        "Values in the model's output were treated as missing, as were those of the cell thickness: "
        + "; ".join(f"{what(value)} in {sim}" for sim, value in missing.items()) + "."
    )
shifted = {sim["name"]: sim["time_shift"] for sim in info["simulations"] if sim["time_shift"]}
if shifted:
    paragraphs.append(
        "The model's times were moved before they were used, as they mark the end of the period each mean covers: "
        + "; ".join(f"{sim} by {days} day{'s' if abs(days) != 1 else ''}" for sim, days in shifted.items()) + "."
    )
md("\n\n".join(paragraphs))

In [ ]:
chunk_end